# Kusanagi — Kaggle full flow

Pipeline: **preflight → train → render 20 train views → metrics → render test → ZIP submission**.

Các scene được chạy tuần tự nên không cộng dồn VRAM. Notebook không backup source và chỉ lưu checkpoint cuối.

In [ ]:
from pathlib import Path
import csv
import gc
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

# ===== PATHS =====
REPO_DIR = Path('/kaggle/working/round2_kusanagi')
DATA_ROOT = Path('/kaggle/input/datasets/acomingzzz/maindataset')
OUTPUT_ROOT = REPO_DIR / 'output_noappearance_radial_hq'
SUBMISSION_DIR = Path('/kaggle/working/submission')
SUBMISSION_ZIP = Path('/kaggle/working/final_submission.zip')

# output name -> folder name inside DATA_ROOT
# Sửa mapping nếu tên folder thực tế trên Kaggle khác (đặc biệt HCM0521).
SCENES = {
    'chair': 'chair',
    'bonsai': 'bonsai',
    'HCM0421': 'HCM0421',
    'HCM0539': 'HCM0539',
    'HCM0540': 'HCM0540',
    'HCM0644': 'HCM0644',
    'HCM0674': 'HCM0674',
}

# ===== TRAINING =====
TRAIN_PROFILE = 'baseline'       # 'baseline' hoặc 'improved'
ITERATIONS = 30_000
TRAIN_RESOLUTION = 1             # đổi thành 2 nếu OOM
RENDER_RESOLUTION = 1            # giữ 1 để ảnh submission đúng kích thước gốc
DATA_DEVICE = 'cpu'              # giảm mạnh VRAM; mỗi iteration chuyển 1 ảnh lên GPU
GPU = '0'
METRIC_TRAIN_VIEWS = 20
METRIC_SEED = 42
SKIP_FINISHED_TRAIN = True
CLEAN_EVAL_IMAGES_AFTER_METRICS = True
APPEARANCE_DIM = 0              # test UID khong map voi train camera
USE_GAUSSIANPRO = False          # A/B validation cho thay GP hien tai khong tang diem
CORRECT_RADIAL_DISTORTION = True
GAUSSIANPRO_DOWNSAMPLE = 2       # doi thanh 4 neu can train nhanh/it VRAM hon

LOSS_PROFILES = {
    'baseline': [],
    'improved': [
        '--use_charbonnier',
        '--lambda_dssim', '0.3',
        '--lambda_lpips', '0.05',
        '--lambda_freq', '0.01',
        '--lpips_net', 'alex',    # alex ít VRAM hơn vgg
        '--lpips_start_iter', '1000',
    ],
}
GAUSSIANPRO_ARGS = [
    '--use_gaussianpro',
    '--gaussianpro_start_iter', '1000',
    '--gaussianpro_interval', '4',
    '--gaussianpro_downsample', str(GAUSSIANPRO_DOWNSAMPLE),
    '--lambda_gaussianpro_flatten', '10.0',
    '--lambda_gaussianpro_normal', '0.05',
    '--lambda_gaussianpro_depth_smooth', '0.01',
] if USE_GAUSSIANPRO else []
RADIAL_ARGS = ['--correct_radial_distortion'] if CORRECT_RADIAL_DISTORTION else []

assert TRAIN_PROFILE in LOSS_PROFILES
os.chdir(REPO_DIR)
os.environ['CUDA_VISIBLE_DEVICES'] = GPU
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Repo:', REPO_DIR)
print('Data:', DATA_ROOT)
print('Profile:', TRAIN_PROFILE)

In [ ]:
# Preflight: GPU, disk và cấu trúc dataset
subprocess.run(['nvidia-smi'], check=False)

def disk_status():
    usage = shutil.disk_usage('/kaggle/working')
    gib = 1024 ** 3
    print(f'Disk /kaggle/working: used={usage.used/gib:.2f} GiB, free={usage.free/gib:.2f} GiB')
    return usage.free

def resolve_train_root(scene_root: Path) -> Path:
    if (scene_root / 'sparse').exists() or (scene_root / 'transforms_train.json').exists():
        return scene_root
    if (scene_root / 'train' / 'sparse').exists() or (scene_root / 'train' / 'transforms_train.json').exists():
        return scene_root / 'train'
    raise FileNotFoundError(f'Không tìm thấy sparse/ hoặc transforms_train.json trong {scene_root}')

available = sorted(p.name for p in DATA_ROOT.iterdir()) if DATA_ROOT.exists() else []
resolved = {}
errors = []
for output_name, folder_name in SCENES.items():
    scene_root = DATA_ROOT / folder_name
    try:
        train_root = resolve_train_root(scene_root)
        images_dir = train_root / 'images'
        image_count = len([p for p in images_dir.iterdir() if p.is_file()]) if images_dir.exists() else 0
        csv_candidates = [scene_root / 'test' / 'test_poses.csv', train_root.parent / 'test' / 'test_poses.csv']
        pose_csv = next((p for p in csv_candidates if p.exists()), None)
        test_count = sum(1 for _ in csv.DictReader(pose_csv.open())) if pose_csv else 0
        resolved[output_name] = scene_root
        print(f'{output_name:10s} train_images={image_count:4d} test_poses={test_count:3d} path={scene_root}')
    except Exception as exc:
        errors.append(f'{output_name}: {exc}')

disk_status()
if errors:
    print('Available DATA_ROOT folders:', available)
    raise RuntimeError('\n'.join(errors))

In [ ]:
# Helpers
def run(cmd, label):
    print('\n' + '=' * 90)
    print(label)
    print(' '.join(map(str, cmd)))
    print('=' * 90, flush=True)
    started = time.time()
    subprocess.run([str(x) for x in cmd], cwd=REPO_DIR, check=True)
    print(f'Finished in {(time.time() - started) / 60:.1f} minutes')
    gc.collect()

def iteration_dir(model_dir: Path) -> Path:
    return model_dir / 'point_cloud' / f'iteration_{ITERATIONS}'

def train_scene(name, source):
    model_dir = OUTPUT_ROOT / name
    if SKIP_FINISHED_TRAIN and (iteration_dir(model_dir) / 'point_cloud.ply').exists():
        print(f'[SKIP TRAIN] {name}: iteration {ITERATIONS} đã tồn tại')
        return
    cmd = [
        sys.executable, 'train.py',
        '-s', source, '-m', model_dir,
        '-r', TRAIN_RESOLUTION,
        '--data_device', DATA_DEVICE,
        '--appearance_dim', APPEARANCE_DIM,
        '--gpu', GPU,
        '--iterations', ITERATIONS,
        '--test_iterations', ITERATIONS,
        '--save_iterations', ITERATIONS,
    ] + LOSS_PROFILES[TRAIN_PROFILE] + GAUSSIANPRO_ARGS + RADIAL_ARGS
    # Không dùng --eval khi train: test poses không có GT sẽ không được nạp.
    run(cmd, f'TRAIN {name}')
    if not (iteration_dir(model_dir) / 'point_cloud.ply').exists():
        raise RuntimeError(f'Train {name} kết thúc nhưng thiếu checkpoint iteration {ITERATIONS}')

def render_metric_views(name, source):
    model_dir = OUTPUT_ROOT / name
    cmd = [
        sys.executable, 'render.py',
        '-s', source, '-m', model_dir,
        '--iteration', ITERATIONS,
        '-r', RENDER_RESOLUTION,
        '--data_device', DATA_DEVICE,
        '--skip_test',
        '--train_views', METRIC_TRAIN_VIEWS,
        '--seed', METRIC_SEED,
    ] + RADIAL_ARGS
    run(cmd, f'RENDER {METRIC_TRAIN_VIEWS} TRAIN VIEWS — {name}')

def calculate_metrics(name):
    model_dir = OUTPUT_ROOT / name
    cmd = [sys.executable, 'metrics.py', '-m', model_dir, '--split', 'train']
    run(cmd, f'METRICS — {name}')
    results_path = model_dir / 'results.json'
    if not results_path.exists():
        raise RuntimeError(f'Metrics không tạo {results_path}')
    result = json.loads(results_path.read_text())
    print(json.dumps(result, indent=2))
    if CLEAN_EVAL_IMAGES_AFTER_METRICS:
        train_render_dir = model_dir / 'train'
        if train_render_dir.exists():
            shutil.rmtree(train_render_dir)
            print('Removed metric render cache:', train_render_dir)

def render_test(name, source):
    model_dir = OUTPUT_ROOT / name
    cmd = [
        sys.executable, 'render.py',
        '-s', source, '-m', model_dir,
        '--iteration', ITERATIONS,
        '-r', RENDER_RESOLUTION,
        '--data_device', DATA_DEVICE,
        '--eval', '--skip_train',
    ] + RADIAL_ARGS
    run(cmd, f'RENDER TEST — {name}')
    render_dir = model_dir / 'test' / f'ours_{ITERATIONS}' / 'renders'
    if not render_dir.exists() or not any(render_dir.iterdir()):
        raise RuntimeError(f'Không có test render tại {render_dir}')
    print(f'{name}: {len(list(render_dir.iterdir()))} test renders')

In [ ]:
# Full pipeline — chạy tuần tự từng scene để VRAM được giải phóng sau mỗi process.
for scene_name, source in resolved.items():
    print(f'\n\n########## {scene_name} ##########')
    disk_status()
    train_scene(scene_name, source)
    render_metric_views(scene_name, source)
    calculate_metrics(scene_name)
    render_test(scene_name, source)

print('\nALL SCENES COMPLETED')
disk_status()

In [ ]:
# Collect và validate submission
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

summary = {}
for scene_name, source in resolved.items():
    render_dir = OUTPUT_ROOT / scene_name / 'test' / f'ours_{ITERATIONS}' / 'renders'
    destination = SUBMISSION_DIR / scene_name
    destination.mkdir(parents=True)
    render_files = sorted(p for p in render_dir.iterdir() if p.is_file())
    for image in render_files:
        shutil.copy2(image, destination / image.name)

    scene_root = Path(source)
    csv_candidates = [scene_root / 'test' / 'test_poses.csv', scene_root.parent / 'test' / 'test_poses.csv']
    pose_csv = next((p for p in csv_candidates if p.exists()), None)
    expected = set()
    if pose_csv:
        with pose_csv.open() as handle:
            expected = {row['image_name'].strip() for row in csv.DictReader(handle)}
    actual = {p.name for p in render_files}
    missing = sorted(expected - actual)
    extra = sorted(actual - expected) if expected else []
    if missing or extra:
        raise RuntimeError(f'{scene_name}: missing={missing}, extra={extra}')
    summary[scene_name] = len(actual)
    print(f'{scene_name:10s}: copied {len(actual)} images')

print('Submission summary:', summary)

In [ ]:
# ZIP giữ cấu trúc: submission/<scene>/<image>
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()
with zipfile.ZipFile(SUBMISSION_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(SUBMISSION_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, Path('submission') / path.relative_to(SUBMISSION_DIR))

with zipfile.ZipFile(SUBMISSION_ZIP) as archive:
    bad = archive.testzip()
    entries = archive.namelist()
if bad:
    raise RuntimeError(f'ZIP corrupted at {bad}')
print(f'DONE: {SUBMISSION_ZIP} ({SUBMISSION_ZIP.stat().st_size / 1024**2:.2f} MiB, {len(entries)} files)')
print('Download file này từ Kaggle Output/Files.')

## Ghi chú

- Nếu improved loss bị CUDA OOM, đổi `TRAIN_RESOLUTION = 2`; vẫn giữ `RENDER_RESOLUTION = 1`.
- `DATA_DEVICE = 'cpu'` tránh giữ toàn bộ ảnh train/test trong VRAM.
- Appearance đã tắt (`APPEARANCE_DIM = 0`); không resume checkpoint cũ đã train với appearance 32 vì kích thước color MLP khác nhau.
- `CORRECT_RADIAL_DISTORTION = True` undistort ảnh HCM khi train và redistort render về camera raw khi xuất submission.
- File `.JPG` được lưu quality 100, chroma 4:4:4; JFIF chỉ là container JPEG bình thường.
- GaussianPro geometry chạy mỗi 4 iteration ở nửa độ phân giải. Nếu chậm/OOM, đặt `GAUSSIANPRO_DOWNSAMPLE = 4` hoặc tăng `--gaussianpro_interval`.
- Không thêm `--eval` vào train. Cờ này chỉ được dùng ở bước render test.
- `SKIP_FINISHED_TRAIN = True` chỉ bỏ qua scene đã có `point_cloud.ply` ở iteration cuối.
- Notebook xóa cache ảnh metric sau khi đã ghi `results.json` và `per_view.json`, nhưng giữ model và test renders.